In [1]:
import warnings
from pathlib import Path

import echopype as ep  # we recommend using "ep"
import xarray as xr
import hvplot.xarray  # for interactive plots

import matplotlib.pyplot as plt

In [2]:
PLOT_SV = True
N_skip = 20

In [3]:
raw_path = '../CRUISES_processed/EBS_2017_AlaskaKnight_leg_4/raw_w_nav/L0531-D20170904-T100303-ES60.raw'

In [4]:
ed = ep.open_raw(raw_path, sonar_model='EK60')

/Users/aflinders/Library/CloudStorage/OneDrive-DOI/NOAA_NCEI_ES60/echopype/echopype/convert/set_groups_ek60.py:710: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'range_sample' ('range_sample',) The recommendation is to set join explicitly for this case.
  [ds, xr.concat(ds_backscatter, dim="channel")], combine_attrs="override"


In [10]:
ed

<EchoData: standardized raw data from Internal Memory>
Top-level: contains metadata about the SONAR-netCDF4 file format.
├── Environment: contains information relevant to acoustic propagation through water.
├── Platform: contains information about the platform on which the sonar is installed.
│   └── NMEA: contains information specific to the NMEA protocol.
├── Provenance: contains metadata about how the SONAR-netCDF4 version of the data were obtained.
├── Sonar: contains sonar system metadata and sonar beam groups.
│   └── Beam_group1: contains backscatter power (uncalibrated) and other beam or channel-specific data, including split-beam angle data when they exist.
└── Vendor_specific: contains vendor-specific information about the sonar and the data.

In [5]:
channels_available = ed["Sonar/Beam_group1"]["channel"].values

In [6]:
# We can then proceed to compute Sv from the raw data. Here, we
# use the calibration and environmental parameters already stored
# in the original instrument-generated .raw file.
#
# Echopype also supports updating these parameters as part of the
# function call, or using an Echoview .ECS file. Note that this
# functionality is in beta and may not cover all equivalent
# vocabulary recognized by Echoview.
#
# compute_Sv requires two arguments: waveform_mode and
# encode_mode, because EK80 files can contain data from a variety
# of sampling configurations, including:

# Use the correct waveform_mode and encode_mode combination
#ds_Sv = ep.calibrate.compute_Sv(ed, waveform_mode="CW", encode_mode="power")
ds_Sv = ep.calibrate.compute_Sv(ed)

# depth_offset=0 for transducer
ds_Sv = ep.consolidate.add_depth(ds_Sv, ed, depth_offset=0)

ds_Sv = ep.consolidate.add_location(ds_Sv, ed)

In [7]:
ds_Sv

<xarray.Dataset> Size: 446MB
Dimensions:                        (channel: 2, ping_time: 1749,
                                    range_sample: 5315, filenames: 1)
Coordinates:
  * channel                        (channel) <U35 280B 'GPT  38 kHz 009072067...
  * ping_time                      (ping_time) datetime64[ns] 14kB 2017-09-04...
  * range_sample                   (range_sample) int64 43kB 0 1 2 ... 5313 5314
  * filenames                      (filenames) int64 8B 0
Data variables: (12/19)
    Sv                             (channel, ping_time, range_sample) float64 149MB ...
    echo_range                     (channel, ping_time, range_sample) float64 149MB ...
    frequency_nominal              (channel) float64 16B 3.8e+04 1.2e+05
    sound_speed                    (channel, ping_time) float64 28kB 1.47e+03...
    sound_absorption               (channel, ping_time) float64 28kB 0.009937...
    sa_correction                  (ping_time, channel) float64 28kB 0.0 ... 0.0
    ...                             ...
    beamwidth_athwartship          (channel) float64 16B 7.1 7.0
    source_filenames               (filenames) <U91 364B '../CRUISES_processe...
    water_level                    float64 8B 0.0
    depth                          (channel, ping_time, range_sample) float64 149MB ...
    latitude                       (ping_time) float64 14kB 54.47 ... 54.41
    longitude                      (ping_time) float64 14kB -166.6 ... -166.6
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.11.1a3.dev111+g8062fbc45
    processing_time:              2026-05-13T04:41:41+00:00
    processing_function:          calibrate.compute_Sv
    processing_level:             Level 2A
    processing_level_url:         https://echopype.readthedocs.io/en/stable/p...

In [ ]:
if PLOT_SV:
    ds_Sv["Sv"].plot(
        x="ping_time", 
        row="channel", col_wrap=3,
        vmin=-80, vmax=-30,
        cmap="RdYlBu_r", yincrease=False
    )

    # Adjust the colourbar range to highlight the seafloor.
    ds_Sv["Sv"].plot(
        x="ping_time", 
        row="channel", col_wrap=3,
        vmin=-40, vmax=-20,
        cmap="RdYlBu_r", yincrease=False,
    )

    #We show the N first samples, to observe the surface saturation zone.
    # (We need to skip them to use the basic seafloor detection.)
    ds_Sv.isel(range_sample=slice(0, 50))["Sv"].plot(
        x="ping_time", 
        row="channel", 
        col_wrap=3,
        vmin=-40, vmax=-20,
        cmap="RdYlBu_r", 
        yincrease=False,
        ylim=(50,0)
    )

In [ ]:
# We use the basic bottom detection function from the seafloor_detection submodule
# within the echopype.mask package.


# work on first channel first
sel_channel = channels_available[0]

from echopype.mask import detect_seafloor
import xarray as xr

# Call detect_seafloor dispatcher
basic_depth = detect_seafloor(
    ds_Sv,
    method="basic",
    params={
        "var_name": "Sv",
        "channel": sel_channel,
        "threshold": (-40, -20),
        "offset_m": 0.3,
        "bin_skip_from_surface": N_skip, # due to surface saturation
    },
)

# Check output
assert isinstance(basic_depth, xr.DataArray)
assert set(basic_depth.dims) == {"ping_time"}

In [ ]:
#We plot the seafloor.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(basic_depth["ping_time"].values, basic_depth.values, fillstyle='full', markersize=1)

ax.set_title("Seafloor depth over ping time")
ax.set_xlabel("Ping time")
ax.set_ylabel("Depth (m)")
ax.invert_yaxis()
plt.show()

In [ ]:
ed["Sonar/Beam_group1"]

In [ ]:
# We compare the results with the Blackwell method, which requires
# the angle_alongship and angle_athwartship variables.

# Extract from EchoData
angle_along = ed["Sonar/Beam_group1"]["angle_alongship"]
angle_athwart = ed["Sonar/Beam_group1"]["angle_athwartship"]

# Merge into ds_Sv
ds_Sv = ds_Sv.assign(
    angle_alongship=angle_along,
    angle_athwartship=angle_athwart
)

required_vars = ["Sv", "angle_alongship", "angle_athwartship", "depth"]
missing = [var for var in required_vars if var not in ds_Sv]
if not missing:
    print("All required variables are present for Blackwell detection.")
else:
    print(f"Missing required variables: {missing}.")

In [ ]:
import matplotlib.pyplot as plt

# Select one channel
angle_athwart_sel = angle_athwart.sel(channel=sel_channel)

angle_athwart_sel.plot(
    x="ping_time",
    y="range_sample",
    cmap="RdBu",
    yincrease=False,
    robust=True,
    cbar_kwargs={"label": "Athwart angle (deg)"}
)
plt.title(f"Channel {sel_channel} – Athwart angle")
plt.show()

In [ ]:
angle_along_sel = angle_along.sel(channel=sel_channel)

angle_along_sel.plot(
    x="ping_time",
    y="range_sample",
    cmap="RdBu",
    yincrease=False,
    robust=True,
    cbar_kwargs={"label": "Alongship angle (deg)"}
)
plt.title(f"Channel {sel_channel} – Alongship angle")
plt.show()

In [ ]:
import numpy as np
import pandas as pd

for name, arr in {
    "angle_alongship": angle_along_sel.values,
    "angle_athwartship": angle_athwart_sel.values,
}.items():
    vals = arr[np.isfinite(arr)]

    print(f"\n{name}")
    print(f"count: {vals.size}")
    print(f"min: {np.min(vals)}")
    print(f"max: {np.max(vals)}")
    print(f"mean: {np.mean(vals)}")
    print(f"std: {np.std(vals)}")
    print(pd.Series(vals).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


In [ ]:
blackwell_depth = detect_seafloor(
    ds=ds_Sv,
    method="blackwell",
    params={
        "channel": sel_channel,
        "var_name": "Sv",
        "threshold": [-40, 702, 282],
        "offset": 0.3,
        "r0": 10,
        "r1": 1000,
        "wtheta": 28,
        "wphi": 52,
    }
)

In [ ]:
# Check output
assert isinstance(blackwell_depth, xr.DataArray)
assert set(blackwell_depth.dims) == {"ping_time"}
blackwell_depth

In [ ]:
# We compare the results from both seafloor detection methods.

# Get the ping_time values from each DataArray
pt_basic = basic_depth.ping_time
pt_blackwell = blackwell_depth.ping_time

# Find ping times in basic_depth but not in blackwell_depth
missing_in_blackwell = pt_basic[~pt_basic.isin(pt_blackwell)]
print(f"Ping times in basic_depth but missing in blackwell_depth: {missing_in_blackwell.size}")
print(missing_in_blackwell.values)

# Find ping times in blackwell_depth but not in basic_depth
missing_in_basic = pt_blackwell[~pt_blackwell.isin(pt_basic)]
print(f"Ping times in blackwell_depth but missing in basic_depth: {missing_in_basic.size}")
print(missing_in_basic.values)

In [ ]:
print("bd.ping_time:", basic_depth)#.ping_time)
print("\n\n")
print("bw.ping_time:", blackwell_depth)#.ping_time)


In [ ]:
# Align both bottom detections on ping_time using xarray
bd, bw = xr.align(basic_depth, blackwell_depth, join="inner")
assert bd.ping_time.equals(bw.ping_time), "Ping times are not aligned."

# Compute difference
diff = bd - bw
common_time = bd.ping_time  # aligned time axis, pick any

# plot
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), sharex=True)

# p1: depth from both methods
axs[0].plot(common_time, bd, label="Basic", color="navy")
axs[0].plot(common_time, bw, label="Blackwell", color="firebrick", linestyle="--")
axs[0].invert_yaxis()
axs[0].set_title("Seafloor depth over ping time")
axs[0].set_xlabel("Ping time")
axs[0].set_ylabel("Depth (m)")
axs[0].legend()
axs[0].grid(True)

# p2: difference
axs[1].plot(common_time, diff, color="darkgreen")
axs[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axs[1].set_title("Difference: Basic – Blackwell")
axs[1].set_xlabel("Ping time")
axs[1].set_ylabel("Depth difference (m)")
axs[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
bd

In [ ]:
ed

In [ ]:
ds_Sv

In [ ]:
ds_Sv["Data variables"]